In [ ]:
GROUP_NAME = "IowaGPT"
NAME = "Amit Boodhoo, Diego Liogon, Eva Singh, Kate Meyer"
PROJECT_NAME = "Project K"

In [ ]:
# imports
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# Sklearn imports for classification
from sklearn import model_selection as skms
from sklearn import metrics
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load the training and testing datasets
train_df = pd.read_csv('train_classification_dataset.csv') 
test_df = pd.read_csv('test_classification_dataset.csv')

# Display basic info to check for missing values and data types
print("---------------------Training Data Info---------------------")
train_df.info()

# show the first 5 rows of the training data
print("---------------------First 5 Rows of Training Dataset---------------------")
display(train_df.head())


# 1. Pre-Analysis Data Considerations
Were there missing values?
How was this addressed?
Was there a linear/non-linear relationship between features/targets?
Were categorical distributions evaluated?
Did they consider the number of rows?
List all of the features, their possible values, and then make a
final column where you comment on if you think the feature will be
useful for predicting the target (your analysis later may prove this wrong).
Below that, write a few comments talking about the features more generally.

In [ ]:
# PART 1: Pre-Analysis Data Exploration

# 1. Check the total number of rows and columns
print("---------------------Dataset Size---------------------")
print(f"Total Rows: {len(train_df)}")
print(f"Total Columns: {len(train_df.columns)}\n")

# 2. Check for missing values
print("--- Missing Values Check ---")
missing_data = train_df.isnull().sum()
print(missing_data[missing_data > 0])
print("\n")

# 3. Check the balance of our Target Variable which is activity_type
print("---------------------Target Variable Balance (activity_type)---------------------")
print(train_df['activity_type'].value_counts())
print("\n")

# 4. Check the distribution of a categorical feature
print("---------------------Feature Distribution (intensity)---------------------")
print(train_df['intensity'].value_counts())

## 1. Pre-Analysis Data Considerations

### Answers to Part 1

**Were there missing values?**
Yes, there were missing values within the `train_classification_dataset.csv`, specifically in the `health_condition` column. We were able to notice this by running a python script that went through the columns to sum up the null values, which showed `health_condition` was missing exactly 147,120 entries. We were also able to notice this manually by visually seeing that there were blank columns denoted by ",," in the raw CSV text. 

**How was this addressed?**
Because the amount of missing data was so huge (over 70% of the column), we could not simply drop those rows without losing most of the dataset. Instead, we filled the missing values with the placeholder text `None` so that the models could still use the rest of each row while preserving the full training set.

**Was there a linear/non-linear relationship between features/targets?**
The relationships appear mostly non-linear and overlapping. Our target (`activity_type`) is a multiclass category, and early inspection suggested that features such as `avg_heart_rate`, `duration_minutes`, and `daily_steps` do shift across activities, but not in a way that cleanly separates all 10 classes with a simple boundary. That is one reason we later tested both distance-based and tree-based classifiers.

**Were categorical distributions evaluated?**
Yes. We checked the observed values in categorical columns such as `intensity`, `gender`, `smoking_status`, and `health_condition`, and we also looked at the class balance of `activity_type`. The target classes are very well balanced, so accuracy is not being inflated by one dominant class.

**Did they consider the number of rows?**
Yes, we checked the size of the dataset and saw that there are 206,310 rows. This is a large dataset, which is helpful because it gives models enough observations to learn patterns and also supports validation methods like 5-fold cross-validation.

### Feature Evaluation Table

| Feature Name | Possible/Observed Values | Expected Usefulness for Predicting Activity Type |
| :--- | :--- | :--- |
| **row_id / participant_id** | Unique Integers | **No.** These are identifiers and do not describe the exercise itself. |
| **date** | YYYY/MM/DD strings | **Yes, after transformation.** The raw date is not directly usable, but month/day features may capture seasonal or calendar effects. |
| **age** | Integers | **Yes.** Age may correlate with the type or intensity of activity chosen. |
| **gender** | 'M', 'F', 'Other' | **Maybe.** Might contain weak but useful demographic signal. |
| **height_cm / weight_kg**| Floats | **Yes.** Physical build can influence exertion and movement patterns. |
| **duration_minutes**| Floats | **Highly Useful.** Helps separate short intense sessions from longer lower-intensity ones. |
| **intensity** | 'Low', 'Medium', 'High' | **Crucial.** Strong signal for distinguishing activities like Yoga from HIIT. |
| **daily_steps** | Integers | **Yes.** Activities involving lots of movement should differ here. |
| **avg_heart_rate** | Integers | **Crucial.** Direct physiological signal of workout exertion. |
| **resting_heart_rate**| Floats | **Maybe.** Baseline fitness marker that may support other features. |
| **blood_pressure_systolic / blood_pressure_diastolic**| Floats | **Maybe.** General health indicators that may add context. |
| **sleep_hours / stress_level** | Floats / Integers | **Maybe.** Lifestyle factors that may weakly influence workout patterns. |
| **hydration_level** | Floats | **Yes.** May reflect how demanding the activity was. |
| **smoking_status** | Strings | **Maybe.** Background health behavior that could add weak signal. |
| **health_condition**| Strings / NaN | **Maybe.** Potentially informative, but requires careful missing-value handling. |
| **activity_type** | 10 Activity Categories | **TARGET VARIABLE.** This is what our models are predicting. |


In [ ]:
# PART 2: Data Cleaning & Feature Engineering 
# Data Wrangling

print("---------------------Starting Data Cleaning---------------------")

# 1. Handle missing values in the dataset
train_df['health_condition'] = train_df['health_condition'].fillna('None')
print("Filled missing health_condition values with 'None'.")

# Clean categorical text so mapping stays consistent
train_df['intensity'] = train_df['intensity'].astype(str).str.strip().str.capitalize()
train_df['gender'] = train_df['gender'].astype(str).str.strip()
train_df['smoking_status'] = train_df['smoking_status'].astype(str).str.strip().str.capitalize()
train_df['health_condition'] = train_df['health_condition'].astype(str).str.strip().str.capitalize()

# 2. Convert categorical text columns into numbers
train_df['intensity'] = train_df['intensity'].map({'Low': 0, 'Medium': 1, 'High': 2}).fillna(0)
train_df['gender'] = train_df['gender'].map({'M': 0, 'F': 1, 'Other': 2})
train_df['smoking_status'] = train_df['smoking_status'].map({'Never': 0, 'Former': 1, 'Current': 2})
train_df['health_condition'] = train_df['health_condition'].map({'None': 0, 'Asthma': 1, 'Diabetes': 2, 'Hypertension': 3})
print("Converted categorical columns into numeric values.")

# 3. Create lecture-safe engineered features
train_df['bmi'] = train_df['weight_kg'] / ((train_df['height_cm'] / 100) ** 2)
parsed_dates = pd.to_datetime(train_df['date'])
train_df['month'] = parsed_dates.dt.month
train_df['day'] = parsed_dates.dt.day

# --- NEW ADVANCED FEATURES ---
train_df['hr_push'] = train_df['avg_heart_rate'] - train_df['resting_heart_rate']
train_df['steps_per_minute'] = train_df['daily_steps'] / train_df['duration_minutes']
train_df['total_effort'] = train_df['duration_minutes'] * train_df['intensity']

# Update selected features to include the new ones
selected_features = [
    'age', 'gender', 'height_cm', 'weight_kg', 'bmi',
    'duration_minutes', 'intensity', 'daily_steps',
    'avg_heart_rate', 'resting_heart_rate',
    'blood_pressure_systolic', 'blood_pressure_diastolic',
    'sleep_hours', 'stress_level', 'hydration_level',
    'smoking_status', 'health_condition', 'month', 'day',
    'hr_push', 'steps_per_minute', 'total_effort'
]

ftrs = train_df[selected_features].copy()
tgt = train_df['activity_type']

ftrs = ftrs.replace([np.inf, -np.inf], np.nan)
ftrs = ftrs.apply(pd.to_numeric, errors='coerce')
ftrs = ftrs.fillna(ftrs.mean())

print(f"Total missing values remaining in features: {ftrs.isnull().sum().sum()}")
print("\n---------------------Data Cleaning Complete. Ready for modeling.---------------------")
print(f"Final Features Shape: {ftrs.shape}")
display(ftrs.head())


## 2. Feature Engineering Justification

**Feature Selection -- How did you decide which features to use?**  
We kept the biometric and lifestyle features because they are the most directly related to physical activity type. Variables such as `avg_heart_rate`, `duration_minutes`, `intensity`, `daily_steps`, `hydration_level`, `age`, `height_cm`, and `weight_kg` all describe either exertion level or the participant's physical profile, so they are reasonable predictors for whether an activity was something like Yoga, Running, HIIT, or Cycling. We dropped `row_id` and `participant_id` because they are identifiers rather than real physiological measurements. Instead of using the raw `date` string directly, we transformed it into `month` and `day`, which is a simpler lecture-covered way to capture potential calendar effects without using the original date text as a shortcut.

**Missing Values**  
The only feature with major missingness was `health_condition`, which had 147,120 missing values, or about 71% of the training data. We did not drop those rows because doing so would remove most of the dataset. Instead, we filled missing values with `None` so that missingness became its own category and the model could still use the rest of each row. This keeps the full training set while still preserving potentially useful information.

**Encoding Decisions**  
We treated `intensity` as an ordinal feature because `Low`, `Medium`, and `High` have a natural order tied to exercise effort, so mapping them to increasing numeric values preserves useful structure. We mapped `gender`, `smoking_status`, and `health_condition` with simple numeric codes so that the categorical values could be used by the lecture-covered models. This is a straightforward baseline encoding approach that stays consistent with the methods used in class, even though it does not mean those categories are truly ordered in reality.

**Additional Preprocessing**  
We created `bmi` from height and weight because the ratio is often more informative than either measurement alone. After the new lecture material, we also engineered `month` and `day` from the `date` column because those simpler numeric features are easier for our models to use than a raw date string. We reserved standardization for Part 3 because it matters most during model training, especially for KNN.

**Evaluation of Features -- Likely to contribute?**  
We expect the strongest predictors to be `avg_heart_rate`, `intensity`, `duration_minutes`, `daily_steps`, and the date-derived `month` and `day` features because they directly describe how physically demanding the activity was and when it occurred. We expect `age`, `height_cm`, `weight_kg`, `resting_heart_rate`, and `hydration_level` to provide supporting information because they reflect physical condition and exercise context. Features such as `sleep_hours`, `stress_level`, `smoking_status`, and `health_condition` may contribute less strongly on their own, but they could still help the model distinguish between activities in borderline cases.


## 3. Model Setup and Strategy

**Train/Test Split & Cross-Validation**
Before training, we split our data using an 80/20 train/test split. We also used the `stratify=tgt` parameter to ensure that the 10 different activity types stayed balanced in both the training set and the validation set. During training, we used 5-fold cross-validation (`cv=5`) so our model comparisons were based on repeated data splits rather than one lucky result.

**Standardization**
After the new lecture material, `StandardScaler()` and `make_pipeline()` became valid course methods for us to use. We applied scaling only to the KNN models because distance-based classifiers are very sensitive to feature magnitude. Putting the scaler inside a pipeline keeps the training and validation process clean and avoids leakage during cross-validation.

**Model Selection & Hyperparameter Tuning**
The rubric requires at least 5 models, so we started with a `DummyClassifier` as a baseline to show what happens if the model always predicts the most common class. We then tested `GaussianNB`, two scaled `KNeighborsClassifier` models (`k=5` and `k=11`), and two `DecisionTreeClassifier` models with different `max_depth` values (`10` and `12`). This gave us both a distance-based family and a tree-based family to compare using the lecture-backed tools.

In [ ]:
# PART 3: Model Setup & Splitting 

# check to make sure there are no "not a number"
print("\nNaNs per column in ftrs:")
print(ftrs.isnull().sum())
print("\nTotal NaNs in ftrs:", ftrs.isnull().sum().sum())

# Train/test split
ftrs_train, ftrs_val, tgt_train, tgt_val = skms.train_test_split(ftrs, tgt,
                                                                    test_size=0.2, # 20% validation
                                                                    stratify=tgt, # keeps class balance, guarantees a fair representation of every activity in groups
                                                                    random_state=42) # to get same training and validation sets every time we run the code    

print("80% train, 20% validation split complete")

In [ ]:
# PART 4: Model Training & Evaluation

from sklearn.ensemble import RandomForestClassifier

models_to_try = {
    "Baseline": DummyClassifier(strategy='most_frequent'),
    "Scaled kNN (k=11)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=11)),
    "Decision Tree (depth=12)": DecisionTreeClassifier(max_depth=12, random_state=42),
    "Random Forest (100 Trees)": RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
}

results = {}

print(f"{'Model Name'} | {'Training Accuracy':<10} | {'Validation Accuracy':<10} | {'Cross Validation Accuracy':<10}")
print("-" * 90)

for name, model in models_to_try.items():
    model.fit(ftrs_train, tgt_train)
    train_acc = metrics.accuracy_score(tgt_train, model.predict(ftrs_train))
    val_acc = metrics.accuracy_score(tgt_val, model.predict(ftrs_val))
    cv_scores = skms.cross_val_score(model, ftrs_train, tgt_train, cv=5)
    cv_acc = cv_scores.mean()

    results[name] = {"cv_acc": cv_acc, "model_obj": model}
    print(f"{name} | {train_acc:.4f} | {val_acc:.4f} | {cv_acc:.4f}")

best_model_name = max(results, key=lambda x: results[x]['cv_acc'])
best_model = results[best_model_name]['model_obj']

print(f"\n---------------------Best Model Selection---------------------")
print(f"Winner: {best_model_name} with {results[best_model_name]['cv_acc']:.4f} Cross Validation Accuracy.")


## 4. Results & Conclusion

**Performance Summary** Our updated models performed better after we incorporated the newly covered lecture methods. The strongest results came from the scaled KNN models and the deeper decision tree, which both improved over the earlier near-baseline performance. This suggests that the classification problem is still difficult, but the broader feature set and better model family choice helped us capture more of the structure in the data.

**Did the models perform well?** The best model was `Scaled kNN (k=5)`, with `Decision Tree (depth=12)` finishing very close behind. This outcome makes sense because the activity classes appear to overlap in non-linear ways, and KNN can take advantage of local neighborhoods once the features are standardized. The decision tree also improved once we allowed deeper splits, which suggests there are interaction effects among the biometric and date-derived features.

In the end, we selected the strongest cross-validated model for our final Kaggle predictions. Even though the overall accuracy is still modest for a 10-class problem, the new lecture material let us build a more defensible and better-performing solution than the earlier baseline-only approach.


## 4. Confusion Matrix Notes

The confusion matrix helps show which activities the best model still mixes up most often. We expect lower-intensity classes such as `Yoga` and `Walking` to overlap with each other more than they overlap with high-intensity classes such as `HIIT`. In contrast, activities with stronger heart-rate and movement signatures should be easier for the model to separate.

When we interpret the matrix, we focus less on any one single error and more on the broader pattern of which activity groups remain difficult to distinguish. That helps explain why the overall accuracy is still limited even after improving the model family and feature set.


In [ ]:
# Visualize the results for the best model (Lecture 15)
from sklearn.metrics import ConfusionMatrixDisplay

tgt_pred = best_model.predict(ftrs_val)
cm = metrics.confusion_matrix(tgt_val, tgt_pred)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_model.classes_)
disp.plot(ax=ax, cmap='Blues', xticks_rotation='vertical')
plt.title(f"Confusion Matrix: {best_model_name}")
plt.show()

In [ ]:
# PART 5: Kaggle Submission

print("Preparing Final Kaggle Submission...")

# 1. Apply the exact same data cleaning steps to 'test_df'
test_df['health_condition'] = test_df['health_condition'].fillna('None')
test_df['intensity'] = test_df['intensity'].astype(str).str.strip().str.capitalize()
test_df['gender'] = test_df['gender'].astype(str).str.strip()
test_df['smoking_status'] = test_df['smoking_status'].astype(str).str.strip().str.capitalize()
test_df['health_condition'] = test_df['health_condition'].astype(str).str.strip().str.capitalize()

test_df['intensity'] = test_df['intensity'].map({'Low': 0, 'Medium': 1, 'High': 2}).fillna(0)
test_df['gender'] = test_df['gender'].map({'M': 0, 'F': 1, 'Other': 2})
test_df['smoking_status'] = test_df['smoking_status'].map({'Never': 0, 'Former': 1, 'Current': 2})
test_df['health_condition'] = test_df['health_condition'].map({'None': 0, 'Asthma': 1, 'Diabetes': 2, 'Hypertension': 3})

# 2. Recreate the exact same engineered features used for training
test_df['bmi'] = test_df['weight_kg'] / ((test_df['height_cm'] / 100) ** 2)
parsed_test_dates = pd.to_datetime(test_df['date'])
test_df['month'] = parsed_test_dates.dt.month
test_df['day'] = parsed_test_dates.dt.day

# --- APPLY NEW ADVANCED FEATURES TO TEST SET ---
test_df['hr_push'] = test_df['avg_heart_rate'] - test_df['resting_heart_rate']
test_df['steps_per_minute'] = test_df['daily_steps'] / test_df['duration_minutes']
test_df['total_effort'] = test_df['duration_minutes'] * test_df['intensity']

ftrs_test_final = test_df[selected_features].copy()
ftrs_test_final = ftrs_test_final.replace([np.inf, -np.inf], np.nan)
ftrs_test_final = ftrs_test_final.apply(pd.to_numeric, errors='coerce')
ftrs_test_final = ftrs_test_final.fillna(ftrs.mean())

# 3. Refit the best model on the full labeled dataset before final prediction
print(f"Refitting final model using: {best_model_name}")
best_model.fit(ftrs, tgt)

# 4. Predict the activity types for the Kaggle test set
final_predictions = best_model.predict(ftrs_test_final)

# 5. Format the predictions into a dataframe matching Kaggle's expected format
submission_df = test_df[['row_id']].copy()
submission_df['activity_type'] = final_predictions

# 6. Save to CSV
submission_filename = 'final_submission.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"\nSuccess! Saved Kaggle predictions to '{submission_filename}'.")
display(submission_df.head())
